In [1]:
# Cell 1: Imports and helper functions

import os
import json
import pandas as pd
import numpy as np

# Percentile helper: ignores NaNs and returns None if series is empty
def pct(series, q):
    s = pd.to_numeric(series, errors="coerce").dropna()
    if s.empty:
        return None
    return float(np.percentile(s, q, method="linear"))

# IQR helper: Interquartile Range (Q3 - Q1)
def calc_iqr(series):
    s = pd.to_numeric(series, errors="coerce").dropna()
    if s.empty or len(s) < 2:
        return None
    q1 = pct(s, 25)
    q3 = pct(s, 75)
    if q1 is None or q3 is None:
        return None
    return float(q3 - q1)

# MAD helper: Median Absolute Deviation
def calc_mad(series):
    s = pd.to_numeric(series, errors="coerce").dropna()
    if s.empty or len(s) < 2:
        return None
    med = np.median(s)
    mad = np.median(np.abs(s - med))
    return float(mad)

print("✓ Imports loaded successfully")

✓ Imports loaded successfully


In [2]:
# Cell 2: CONFIG — PASTE FULL PATHS ONLY

# ============================================================
# EDIT THESE THREE LINES - Paste complete paths
# ============================================================

# Database name (for display in output CSV)
db_name = "AstraDB"

# Full path to the INPUT CSV file (reads_A2.csv)
a2_csv_path = r"C:\Users\avyaa\logs\reads_A2.csv"

# Full path to the OUTPUT FOLDER (where you want a2_metrics.csv saved)
# You must create this folder manually before running
a2_out_dir = r"C:\Users\avyaa\Astra_results\A2\Data"

# ============================================================
# DO NOT EDIT BELOW
# ============================================================

# Output file paths
a2_metrics_out = os.path.join(a2_out_dir, "a2_metrics.csv")
a2_validation_out = os.path.join(a2_out_dir, "a2_validation.csv")

# A2 expectations (all warm, random parameters)
EXPECTED_WARM = 10

# Verification
print("Database:", db_name)
print("Input CSV:", a2_csv_path)
print("Output metrics:", a2_metrics_out)
print("Output validation:", a2_validation_out)

if not os.path.exists(a2_csv_path):
    print("\n⚠️ INPUT FILE NOT FOUND! Check your path.")
else:
    print("\n✓ Input file found!")
    
if not os.path.exists(a2_out_dir):
    print("⚠️ OUTPUT FOLDER NOT FOUND! Create it manually first.")
else:
    print("✓ Output folder exists!")

Database: AstraDB
Input CSV: C:\Users\avyaa\logs\reads_A2.csv
Output metrics: C:\Users\avyaa\Astra_results\A2\Data\a2_metrics.csv
Output validation: C:\Users\avyaa\Astra_results\A2\Data\a2_validation.csv

✓ Input file found!
✓ Output folder exists!


In [3]:
# Cell 3: Load A2 CSV, retain A2 rows, and prep columns

# Required columns per logging strategy
required_cols = [
    "timestamp_utc","run_id","db_system","server_version","driver",
    "connection_params","db_name","mode","param_source","operation_type",
    "query_id","complexity","query_name","run_number",
    "execution_ms","row_count","error_code","error_message","params_json"
]

df = pd.read_csv(a2_csv_path)

# Check for missing columns
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

# Keep only A2 reads (mode=A2, param_source=random, operation_type=read)
df = df[(df["mode"] == "A2") & (df["operation_type"] == "read")].copy()

# Normalize numerics
df["run_number"] = pd.to_numeric(df["run_number"], errors="coerce")
df["execution_ms"] = pd.to_numeric(df["execution_ms"], errors="coerce")
df["row_count"] = pd.to_numeric(df["row_count"], errors="coerce")

# Separate views for metrics (exclude error rows with no execution_ms)
df_ok = df[df["execution_ms"].notna()].copy()
df_err = df[df["execution_ms"].isna()].copy()

print("A2 total rows:", len(df))
print("Usable rows (with execution_ms):", len(df_ok))
print("Error rows (no execution_ms):", len(df_err))

# A2 should have NO cold runs - verify
if "is_cold" in df.columns:
    cold_rows = df[df["is_cold"].astype(str).str.lower().isin(["1","true","yes"])]
    if len(cold_rows) > 0:
        print(f"\n⚠️ WARNING: Found {len(cold_rows)} cold runs in A2 data (should be zero)")

A2 total rows: 100
Usable rows (with execution_ms): 100
Error rows (no execution_ms): 0


In [4]:
# Cell 4: Count validation (at least EXPECTED_WARM runs per query, all warm/random)

val_rows = []
for qid, g in df.groupby("query_id"):
    warm_cnt = int(len(g))  # All A2 runs should be warm
    errs     = int(g["execution_ms"].isna().sum())
    
    # Check parameter diversity (params_json should vary across runs)
    params_unique = g["params_json"].nunique() if "params_json" in g.columns else 0
    
    val_rows.append({
        "db_system": db_name,
        "query_id": qid,
        "run_count": warm_cnt,
        "error_rows": errs,
        "unique_params": params_unique,
        "expected_runs": EXPECTED_WARM,
        "valid_counts": warm_cnt >= EXPECTED_WARM,
        "params_vary": params_unique > 1  # Parameters should vary across runs
    })

val_df = pd.DataFrame(val_rows).sort_values("query_id")
val_df.to_csv(a2_validation_out, index=False)

print("✓ Validation saved to:", a2_validation_out)
print("\nValidation summary:")
display(val_df)

✓ Validation saved to: C:\Users\avyaa\Astra_results\A2\Data\a2_validation.csv

Validation summary:


,db_system,query_id,run_count,error_rows,unique_params,expected_runs,valid_counts,params_vary
0,AstraDB,R1,10,0,10,10,True,True
1,AstraDB,R10,10,0,10,10,True,True
2,AstraDB,R2,10,0,10,10,True,True
3,AstraDB,R3,10,0,10,10,True,True
4,AstraDB,R4,10,0,9,10,True,True
5,AstraDB,R5,10,0,10,10,True,True
6,AstraDB,R6,10,0,1,10,True,False
7,AstraDB,R7,10,0,10,10,True,True
8,AstraDB,R8,10,0,1,10,True,False
9,AstraDB,R9,10,0,8,10,True,True


In [5]:
# Cell 5: A2 percentiles per query (p50, p95, p99, max) - all warm/random

agg_a2 = []
for qid, g in df_ok.groupby(["query_id","complexity","query_name"]):
    p50 = pct(g["execution_ms"], 50)
    p95 = pct(g["execution_ms"], 95)
    p99 = pct(g["execution_ms"], 99)
    mx  = float(np.max(g["execution_ms"])) if len(g) else None
    
    agg_a2.append({
        "query_id": qid[0],
        "complexity": qid[1],
        "query_name": qid[2],
        "a2_p50_ms": p50,
        "a2_p95_ms": p95,
        "a2_p99_ms": p99,
        "a2_max_ms": mx,
        "run_count": int(len(g))
    })

a2_df = pd.DataFrame(agg_a2).sort_values("query_id")

print("✓ A2 percentiles computed")
display(a2_df.head(10))

✓ A2 percentiles computed


,query_id,complexity,query_name,a2_p50_ms,a2_p95_ms,a2_p99_ms,a2_max_ms,run_count
0,R1,Basic,Product by ID,190.3115,194.37425,195.09965,195.281,10
1,R10,Advanced,Attribute-like filter (client-side sort),568.4970,1402.44740,1469.57588,1486.358,10
2,R2,Basic,Product by SKU,186.9130,201.09795,203.70039,204.351,10
3,R3,Basic,Customer order history,201.1925,211.86565,217.48993,218.896,10
4,R4,Basic,Category browse,196.5385,203.39085,205.71897,206.301,10
5,R5,Moderate,Search with filters (equality + range),189.0650,200.27465,201.53573,201.851,10
6,R6,Moderate,Best Sellers 30 Day,386350.4750,388073.03835,388468.92927,388567.902,10
7,R7,Moderate,New arrivals by window,193.1965,198.43405,201.05881,201.715,10
8,R8,Moderate,Order detail with items,384.7140,396.20590,398.75398,399.391,10
9,R9,Advanced,Brand facet count (client-side),208.8520,216.10685,216.33257,216.389,10


In [6]:
# Cell 6: Dispersion indicators per query (IQR and MAD)

disp_rows = []
for qid, g in df_ok.groupby(["query_id","complexity","query_name"]):
    iqr_val = calc_iqr(g["execution_ms"])
    mad_val = calc_mad(g["execution_ms"])
    
    disp_rows.append({
        "query_id": qid[0],
        "complexity": qid[1],
        "query_name": qid[2],
        "iqr_ms": iqr_val,
        "mad_ms": mad_val
    })

disp_df = pd.DataFrame(disp_rows).sort_values("query_id")

print("✓ Dispersion indicators computed")
display(disp_df.head(10))

✓ Dispersion indicators computed


,query_id,complexity,query_name,iqr_ms,mad_ms
0,R1,Basic,Product by ID,2.08375,1.1325
1,R10,Advanced,Attribute-like filter (client-side sort),866.32150,369.2115
2,R2,Basic,Product by SKU,7.49200,2.1925
3,R3,Basic,Customer order history,4.92275,2.2935
4,R4,Basic,Category browse,4.66350,2.7660
5,R5,Moderate,Search with filters (equality + range),10.18700,4.5990
6,R6,Moderate,Best Sellers 30 Day,1246.54325,735.9605
7,R7,Moderate,New arrivals by window,5.77500,1.7040
8,R8,Moderate,Order detail with items,4.86725,3.0045
9,R9,Advanced,Brand facet count (client-side),7.77825,4.6640


In [7]:
# Cell 7: Tail amplification on A2 runs (p99/p50)

tail_rows = []
for qid, g in df_ok.groupby(["query_id","complexity","query_name"]):
    p50 = pct(g["execution_ms"], 50)
    p99 = pct(g["execution_ms"], 99)
    
    tail_amp = None
    if p50 is not None and p50 > 0 and p99 is not None:
        tail_amp = float(p99 / p50)
    
    tail_rows.append({
        "query_id": qid[0],
        "complexity": qid[1],
        "query_name": qid[2],
        "tail_amp_p99_p50": tail_amp
    })

tail_df = pd.DataFrame(tail_rows).sort_values("query_id")

print("✓ Tail amplification computed")
display(tail_df.head(10))

✓ Tail amplification computed


,query_id,complexity,query_name,tail_amp_p99_p50
0,R1,Basic,Product by ID,1.025160
1,R10,Advanced,Attribute-like filter (client-side sort),2.585020
2,R2,Basic,Product by SKU,1.089814
3,R3,Basic,Customer order history,1.081004
4,R4,Basic,Category browse,1.046711
5,R5,Moderate,Search with filters (equality + range),1.065960
6,R6,Moderate,Best Sellers 30 Day,1.005483
7,R7,Moderate,New arrivals by window,1.040696
8,R8,Moderate,Order detail with items,1.036495
9,R9,Advanced,Brand facet count (client-side),1.035818


In [8]:
# Cell 8: Merge all metrics and create final output

# Start with percentiles
metrics = a2_df.copy()

# Merge dispersion
metrics = pd.merge(metrics, disp_df, on=["query_id","complexity","query_name"], how="left")

# Merge tail amplification
metrics = pd.merge(metrics, tail_df.loc[:, ["query_id","tail_amp_p99_p50"]], on="query_id", how="left")

# Merge validation info
metrics = pd.merge(
    metrics,
    val_df.loc[:, ["query_id","error_rows","unique_params","valid_counts","params_vary"]],
    on="query_id",
    how="left"
)

# Attach db_system for clarity
metrics.insert(0, "db_system", db_name)

# Order columns
col_order = [
    "db_system","query_id","complexity","query_name",
    "run_count","error_rows","unique_params","valid_counts","params_vary",
    "a2_p50_ms","a2_p95_ms","a2_p99_ms","a2_max_ms",
    "iqr_ms","mad_ms","tail_amp_p99_p50"
]
metrics = metrics.loc[:, col_order]

# Save
metrics.to_csv(a2_metrics_out, index=False)

print("✓ Metrics saved to:", a2_metrics_out)
print("\nFinal A2 metrics:")
display(metrics)

✓ Metrics saved to: C:\Users\avyaa\Astra_results\A2\Data\a2_metrics.csv

Final A2 metrics:


,db_system,query_id,complexity,query_name,run_count,error_rows,unique_params,valid_counts,params_vary,a2_p50_ms,a2_p95_ms,a2_p99_ms,a2_max_ms,iqr_ms,mad_ms,tail_amp_p99_p50
0,AstraDB,R1,Basic,Product by ID,10,0,10,True,True,190.3115,194.37425,195.09965,195.281,2.08375,1.1325,1.025160
1,AstraDB,R10,Advanced,Attribute-like filter (client-side sort),10,0,10,True,True,568.4970,1402.44740,1469.57588,1486.358,866.32150,369.2115,2.585020
2,AstraDB,R2,Basic,Product by SKU,10,0,10,True,True,186.9130,201.09795,203.70039,204.351,7.49200,2.1925,1.089814
3,AstraDB,R3,Basic,Customer order history,10,0,10,True,True,201.1925,211.86565,217.48993,218.896,4.92275,2.2935,1.081004
4,AstraDB,R4,Basic,Category browse,10,0,9,True,True,196.5385,203.39085,205.71897,206.301,4.66350,2.7660,1.046711
5,AstraDB,R5,Moderate,Search with filters (equality + range),10,0,10,True,True,189.0650,200.27465,201.53573,201.851,10.18700,4.5990,1.065960
6,AstraDB,R6,Moderate,Best Sellers 30 Day,10,0,1,True,False,386350.4750,388073.03835,388468.92927,388567.902,1246.54325,735.9605,1.005483
7,AstraDB,R7,Moderate,New arrivals by window,10,0,10,True,True,193.1965,198.43405,201.05881,201.715,5.77500,1.7040,1.040696
8,AstraDB,R8,Moderate,Order detail with items,10,0,1,True,False,384.7140,396.20590,398.75398,399.391,4.86725,3.0045,1.036495
9,AstraDB,R9,Advanced,Brand facet count (client-side),10,0,8,True,True,208.8520,216.10685,216.33257,216.389,7.77825,4.6640,1.035818


In [9]:
# Cell 9: Sanity checks and summaries

print("=" * 60)
print("A2 METRICS COMPUTATION COMPLETE")
print("=" * 60)

print("\n📁 Files saved:")
print("  -", a2_metrics_out)
print("  -", a2_validation_out)

print("\n⚠️ Queries missing expected counts:")
invalid = val_df[~val_df["valid_counts"]]
if len(invalid) > 0:
    display(invalid)
else:
    print("  ✓ All queries have valid counts!")

print("\n⚠️ Queries with non-varying parameters:")
no_vary = val_df[~val_df["params_vary"]]
if len(no_vary) > 0:
    display(no_vary)
else:
    print("  ✓ All queries have varying parameters!")

print("\n📊 Highest dispersion (top 5 by IQR):")
display(metrics.sort_values("iqr_ms", ascending=False).head(5))

print("\n🔥 Largest tail amplification (top 5):")
display(metrics.sort_values("tail_amp_p99_p50", ascending=False).head(5))

A2 METRICS COMPUTATION COMPLETE

📁 Files saved:
  - C:\Users\avyaa\Astra_results\A2\Data\a2_metrics.csv
  - C:\Users\avyaa\Astra_results\A2\Data\a2_validation.csv

⚠️ Queries missing expected counts:
  ✓ All queries have valid counts!

⚠️ Queries with non-varying parameters:


,db_system,query_id,run_count,error_rows,unique_params,expected_runs,valid_counts,params_vary
6,AstraDB,R6,10,0,1,10,True,False
8,AstraDB,R8,10,0,1,10,True,False



📊 Highest dispersion (top 5 by IQR):


,db_system,query_id,complexity,query_name,run_count,error_rows,unique_params,valid_counts,params_vary,a2_p50_ms,a2_p95_ms,a2_p99_ms,a2_max_ms,iqr_ms,mad_ms,tail_amp_p99_p50
6,AstraDB,R6,Moderate,Best Sellers 30 Day,10,0,1,True,False,386350.475,388073.03835,388468.92927,388567.902,1246.54325,735.9605,1.005483
1,AstraDB,R10,Advanced,Attribute-like filter (client-side sort),10,0,10,True,True,568.497,1402.44740,1469.57588,1486.358,866.32150,369.2115,2.585020
5,AstraDB,R5,Moderate,Search with filters (equality + range),10,0,10,True,True,189.065,200.27465,201.53573,201.851,10.18700,4.5990,1.065960
9,AstraDB,R9,Advanced,Brand facet count (client-side),10,0,8,True,True,208.852,216.10685,216.33257,216.389,7.77825,4.6640,1.035818
2,AstraDB,R2,Basic,Product by SKU,10,0,10,True,True,186.913,201.09795,203.70039,204.351,7.49200,2.1925,1.089814



🔥 Largest tail amplification (top 5):


,db_system,query_id,complexity,query_name,run_count,error_rows,unique_params,valid_counts,params_vary,a2_p50_ms,a2_p95_ms,a2_p99_ms,a2_max_ms,iqr_ms,mad_ms,tail_amp_p99_p50
1,AstraDB,R10,Advanced,Attribute-like filter (client-side sort),10,0,10,True,True,568.4970,1402.44740,1469.57588,1486.358,866.32150,369.2115,2.585020
2,AstraDB,R2,Basic,Product by SKU,10,0,10,True,True,186.9130,201.09795,203.70039,204.351,7.49200,2.1925,1.089814
3,AstraDB,R3,Basic,Customer order history,10,0,10,True,True,201.1925,211.86565,217.48993,218.896,4.92275,2.2935,1.081004
5,AstraDB,R5,Moderate,Search with filters (equality + range),10,0,10,True,True,189.0650,200.27465,201.53573,201.851,10.18700,4.5990,1.065960
4,AstraDB,R4,Basic,Category browse,10,0,9,True,True,196.5385,203.39085,205.71897,206.301,4.66350,2.7660,1.046711
